# Semantic Search using PineCone DB (Cloud Based)

website: https://www.pinecone.io/

## Objective

1) Convert text → embeddings

2) Store in vector database

3) Perform semantic search

In [1]:
# Following too long time
!pip install pinecone sentence-transformers

# SO I used following on  command line: Took 15 minutes
# python -m pip install pinecone sentence-transformers

  Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl.metadata (1.2 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
   ---------------------------------------- 0.0/742.7 kB ? eta -:--:--
   ---------------------------------------- 742.7/742.7 kB 7.6 MB/s  0:00:00
Using cached packaging-24.2-py3-none-any.whl (65 kB)
Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl (6.2 kB)

   ---------------------------------------- 0/5 [pinecone-plugin-interface]
  Attempting uninstall: packaging
   ---------------------------------------- 0/5 [pinecone-plugin-interface]
    Found existing installation: packaging 25.0
   ---------------------------------------- 0/5 [pinecone-plugin-interface]
    Uninstalling packaging-25.0:
   ---------------------------------------- 0/5 [pinecone-plugin-interface]
      Successfully uninstalled packaging-25.0
   ---------------------------------------- 0/5 [pinecone-plugin-interface]
   -------- ---------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.4.3 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Import Librarairs: Took a 4-5 minutes
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

In [3]:
# Need API Key
pc = Pinecone(api_key="pcsk_key")
# pc = Pinecone(api_key="pcsk_KEY")

print("🔄 Testing Pinecone API Key connectivity...")

try:
    # 2. Trigger a control plane API call to verify the key
    active_indexes = pc.list_indexes()
    
    print("✅ Success! Your Pinecone API key is valid and authenticated.")
    print(f"📁 Available Indexes in your project: {len(active_indexes)}")
    for index in active_indexes:
        print(f"   - {index.name} ({index.dimension}d, {index.metric})")

except PineconeException as e:
    # 3. Handle specific authentication/network errors safely
    print("❌ Authentication Failed!")
    print(f"🔍 Error details: {str(e)}")

🔄 Testing Pinecone API Key connectivity...
✅ Success! Your Pinecone API key is valid and authenticated.
📁 Available Indexes in your project: 1
   - advanced-rag (1536d, cosine)


# Create an index

In [4]:
index_name = "genai-demo"

if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

# Connect to Index

In [5]:
index = pc.Index(index_name)

# Prepare the documents

In [6]:
# Contains 2 main topics: AI and Cars
documents = [
    "Machine learning is a subset of AI",
    "Deep learning uses neural networks",
    "Python is used for data science",
    "Cars and vehicles are transportation",
    "Artificial intelligence is transforming industries",
    "Football is a popular sport",
    "Data engineering involves pipelines",
    "Rides are good mode of transportation",
    "Cars and automobiles are expensive",
    "Healthcare is improved by AI diagnostics. It is bringing positive change in the world",
    "Self-driving vehicles use machine learning"
]

In [7]:
vectors = [
    {
        "_id": str(i),
        "chunk_text": doc   # must match field_map
    }
    for i, doc in enumerate(documents)
]

print(vectors)

[{'_id': '0', 'chunk_text': 'Machine learning is a subset of AI'}, {'_id': '1', 'chunk_text': 'Deep learning uses neural networks'}, {'_id': '2', 'chunk_text': 'Python is used for data science'}, {'_id': '3', 'chunk_text': 'Cars and vehicles are transportation'}, {'_id': '4', 'chunk_text': 'Artificial intelligence is transforming industries'}, {'_id': '5', 'chunk_text': 'Football is a popular sport'}, {'_id': '6', 'chunk_text': 'Data engineering involves pipelines'}, {'_id': '7', 'chunk_text': 'Rides are good mode of transportation'}, {'_id': '8', 'chunk_text': 'Cars and automobiles are expensive'}, {'_id': '9', 'chunk_text': 'Healthcare is improved by AI diagnostics. It is bringing positive change in the world'}, {'_id': '10', 'chunk_text': 'Self-driving vehicles use machine learning'}]


# Store the documents in vector DB

In [9]:
# Upsert the records into a namespace

index.upsert_records(
    namespace="example-namespace", 
    records=vectors
)

UpsertRecordsResponse(record_count=11, response_info=ResponseInfo(raw_headers={'date': 'Fri, 12 Jun 2026 11:39:19 GMT', 'content-length': '0', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '1', 'x-pinecone-api-version': '2025-10', 'x-envoy-upstream-service-time': '743', 'x-pinecone-response-duration-ms': '744', 'server': 'envoy'}))

In [10]:
# Wait for the upserted vectors to be indexed
import time
time.sleep(10)

# View stats for the index
stats = index.describe_index_stats()
print(stats)

DescribeIndexStatsResponse(dimension=1024, total_vector_count=11, metric='cosine', namespaces=1)


# Perform Semantic Search

In [16]:
# Define the query
query = "how is AI is changing the world"

# Search the dense index
results = index.search(
    namespace="example-namespace",
    query={
        "top_k": 10,
        "inputs": {
            'text': query
        }
    }
)

In [17]:
# Returns results sorted by similarity score
print(results)

SearchRecordsResponse(result=SearchResult(hits=[Hit(id='9', score=0.48799553513526917, fields={'chunk_text': 'Healthcare is improved by AI diagnostics. It is bringing positive change in the world'}), Hit(id='4', score=0.3506844639778137, fields={'chunk_text': 'Artificial intelligence is transforming industries'}), Hit(id='10', score=0.17753763496875763, fields={'chunk_text': 'Self-driving vehicles use machine learning'}), Hit(id='0', score=0.16916680335998535, fields={'chunk_text': 'Machine learning is a subset of AI'}), Hit(id='1', score=0.1292077898979187, fields={'chunk_text': 'Deep learning uses neural networks'}), Hit(id='2', score=0.11820488423109055, fields={'chunk_text': 'Python is used for data science'}), Hit(id='6', score=0.06901102513074875, fields={'chunk_text': 'Data engineering involves pipelines'}), Hit(id='3', score=0.05909710004925728, fields={'chunk_text': 'Cars and vehicles are transportation'}), Hit(id='8', score=0.023693891242146492, fields={'chunk_text': 'Cars an

In [18]:
# Print the results
for hit in results['result']['hits']: # returns result sorted by score
    # print(hit)
    print(f"id: {hit['id']:<5} | score: {round(hit['score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")


id: 9     | score: 0.49  | text: Healthcare is improved by AI diagnostics. It is bringing positive change in the world
id: 4     | score: 0.35  | text: Artificial intelligence is transforming industries
id: 10    | score: 0.18  | text: Self-driving vehicles use machine learning        
id: 0     | score: 0.17  | text: Machine learning is a subset of AI                
id: 1     | score: 0.13  | text: Deep learning uses neural networks                
id: 2     | score: 0.12  | text: Python is used for data science                   
id: 6     | score: 0.07  | text: Data engineering involves pipelines               
id: 3     | score: 0.06  | text: Cars and vehicles are transportation              
id: 8     | score: 0.02  | text: Cars and automobiles are expensive                
id: 5     | score: 0.01  | text: Football is a popular sport                       


In [19]:
# Lets use a different query: on  cars
query = "Tell me about rides ?"

# Search the dense index
results = index.search(
    namespace="example-namespace",
    query={
        "top_k": 10,
        "inputs": {
            'text': query
        }
    }
)

# Print the results
for hit in results['result']['hits']: # returns result sorted by score
    # print(hit)
    print(f"id: {hit['id']:<5} | score: {round(hit['score'], 2):<5} | text: {hit['fields']['chunk_text']:<50}")


id: 7     | score: 0.26  | text: Rides are good mode of transportation             
id: 3     | score: 0.08  | text: Cars and vehicles are transportation              
id: 8     | score: 0.05  | text: Cars and automobiles are expensive                
id: 5     | score: 0.04  | text: Football is a popular sport                       
id: 10    | score: 0.01  | text: Self-driving vehicles use machine learning        
id: 9     | score: -0.01 | text: Healthcare is improved by AI diagnostics. It is bringing positive change in the world
id: 4     | score: -0.02 | text: Artificial intelligence is transforming industries
id: 6     | score: -0.03 | text: Data engineering involves pipelines               
id: 0     | score: -0.03 | text: Machine learning is a subset of AI                
id: 1     | score: -0.04 | text: Deep learning uses neural networks                


# Delete the index when done

In [20]:
pc.delete_index(index_name)